In [ ]:
import praw
import streamlit as st
import pandas as pd
import numpy as np
from openai import OpenAI
import re
import json

In [ ]:
client = OpenAI(api_key="YOUR-API-KEY")

## Validating if the comments/arguments are in JSON format

In [ ]:
def is_valid_json(json_str):
    try:
        json_object = json.loads(json_str)
    except ValueError as e:
        return False, str(e)
    return True, json_object

def handle_response(response_text):
    # Remove any extraneous text or formatting
    response_text = response_text.strip()
    if response_text.startswith('```json'):
        response_text = response_text[7:]
    if response_text.endswith('```'):
        response_text = response_text[:-3]
    response_text = response_text.strip()
    # Validate JSON
    is_valid, result = is_valid_json(response_text)
    if is_valid:
        return result
    else:
        raise ValueError(f"Invalid JSON response: {result}")

## CMV URL Used for the Project

In [ ]:
url  = "https://www.reddit.com/r/changemyview/comments/1dq92v2/cmv_this_current_presidential_debate_has_proved/"

In [ ]:
##Scraps a list of comments from Reddit CMV and then sends an API call to OpenAI in order to determine the epl scores of those comments.

def fetch_comments_with_scores(post_url):
    reddit = praw.Reddit(
        client_id='YOUR-CLIENT-ID',
        client_secret='YOUR-SECRET-ID',
        user_agent='YOUR-USER-AGENT'
    )

    submission = reddit.submission(url=post_url)
    submission.comments.replace_more(limit=None)

    comments = []
    #The list is used to set range of comments collected
    for comment in submission.comments.list()[1:6]:
        if not comment.author:  # Check if comment is deleted (author is None)
            continue

        comments.append({
            'author': comment.author.name,
            'argument_text': json.dumps(comment.body)
        })

    json_output = {"comments_with_scores": []}

    for idx, comment_row in enumerate(comments):
        comment_text = json.loads(comment_row['argument_text'])

        comment_data = {
            "Comment Number": idx + 1,
            "Author Name": comment_row['author'],
            "Comment Text": comment_text,
        }
        json_output["comments_with_scores"].append(comment_data)

    return json_output

In [ ]:
#comment_json = fetch_comments_with_scores(url)

#comment_df = pd.DataFrame(comment_json)

In [ ]:
##OpenAI function to gather EPL scores

def getEPLAverages(df):
    epl = []
    #for row in df.itertuples():
    for idx, row in df.iterrows():
        system_prompt_epl_scores = """You are an AI Rhetorical Analysis Expert. Determine the ethos (e), pathos (p), and logos (l) scores for each turn. For each attribute, score them on a scale of [0, 1) and provide exactly 1-2 sentence explainer (expl). Return a single RFC 8259 compliant JSON object:
        {"ethos": int,
        "pathos": int,
        "logos": int,
        "e_expl": String,
        "p_expl": String,
        "l_expl": String
        }"""
        user_content = "Analyze the turn: " + json.dumps(row.to_dict())
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt_epl_scores},
                {"role": "user", "content": user_content}
            ]
        )
        try:
            result = response.choices[0].message.content
            result = handle_response(result)

        except ValueError as e:
            print(e)

        epl.append(result)

    return epl

In [ ]:
#epl = getEPLAverages(comment_df)

#epl_df = pd.json_normalize(epl)

#epl_df.to_pickle("epl_comments-.pkl")

#comment_df = comment_df.join(epl_df, how="outer")

In [ ]:
##Loading pickle files and then using the EPLAverages function to give the scores to the pickle files.
##Then, storing that into a different pickle file called “epl_comments_.pkl”

def load_from_pickle(output_directory, pickle_filename):
    # Construct path to the pickle file
    pickle_path = os.path.join(output_directory, pickle_filename)
    try:
        with open(pickle_path, 'rb') as file:
            data = pickle.load(file)
        return data
    except FileNotFoundError:
        print(f"Error: File '{pickle_filename}' not found in '{output_directory}'")
        return None
    except Exception as e:
        print(f"Error loading pickle file: {e}")
        return None

In [ ]:
##Creates a Radar Plot based off of the dataframe of EPL scores across a comment section, and the number of traces the user wishes to be visualized (implemented with a slider in Streamlit).
##Determines which score is greatest for each comment, and color codes it accordingly (green for ethos, red for pathos, blue for logos).

def eplRadarPlot(df, num_traces):
   fig = go.Figure()

   categories = ["ethos", "pathos", "logos"]
   category_colors = {
       "ethos": '#8B9DCE',
       "pathos": '#F0B87F',
       "logos": '#9FBC8C'
   }
   color_descriptions = {
       "ethos": "light blue",
       "pathos": "light orange",
       "logos": "light green"
   }

   color_counts = {category: 0 for category in categories}

   def get_color(row):
       max_category = row[categories].idxmax()
       return category_colors[max_category]

   def get_dominant_category_name(row):
       max_category = row[categories].idxmax()
       return f'{max_category}-heavy'

   for index, row in df.head(num_traces).iterrows():
       color = get_color(row)
       max_category = row[categories].idxmax()
       color_counts[max_category] += 1

       trace_name = get_dominant_category_name(row)
       hover_info = '<br>'.join([f'{category}: {row[category]}' for category in categories])
       fig.add_trace(go.Scatterpolar(
             r=row[categories],
             theta=categories,
             fill='toself',
             name=trace_name,
             line=dict(color=color),
             hovertext=hover_info,
             hoverinfo='text'
       ))

       annotation_text = '<br>'.join([f'{category} - {color_descriptions[category]}: {count} dominant trace(s)' for category, count in color_counts.items()])

       fig.update_layout(
         polar=dict(
           radialaxis=dict(
             visible=True,
             range=[0, 10]
           ),
           bgcolor='rgba(255, 255, 255, 0.3)'
         ),
         annotations=[
           dict(
               xref='paper',
               yref='paper',
               x=1.2,
               y=-0.3,
               text=annotation_text,
               showarrow=False,
               font=dict(size=12),
               align='left'
           )
         ]
       )

   return fig


#st.title("EPL Scores Radar Plot")

#num_traces = st.slider('The minimum end of the range is more likely to show traces for the top/most upvoted comments. The maximum end of the range is more likely to show the bottom/least upvoted comments. Select the number of traces to display:', min_value=5, max_value=len(reddit_df), value=10, step=5)

#radar_plot = eplRadarPlot(reddit_df, num_traces)
#st.plotly_chart(radar_plot)


In [ ]:
##Description: Makes a heatmap for all ethos, pathos, logos across the comments. The bigger the number, it means more positive correlation (-1, 0, 1).
##Positive means as one attribute increases, the other increases. 0 means no correlation. Negative means as one increases, the other decreases.

def epl_heatmap(df):
   # Compute the correlation matrix
   correlation_matrix = df[['ethos', 'pathos', 'logos']].corr()

   # Create a heatmap
   plt.figure(figsize=(10, 8))
   heatmap = sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
   plt.title('Correlation Heatmap of Ethos, Pathos, and Logos Scores')

   return plt.gcf()

In [ ]:
##Description: Makes a heatmap between the correlation of all the 3 triads and the personas. There are 3 graphs, where one is for pathos, one is for ethos, one is for logos.
##Previously stated, (0-10), that the higher the number, the more concentrated the attributes (ethos, pathos, logos) between a persona and triad.

def triad_persona_heatmap(df):
    # Pivot table to get average scores for each combination of persona and triad
    pivot_table = df.pivot_table(index='personas', columns='triad', values=['ethos', 'pathos', 'logos'], aggfunc='mean')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    for i, score_type in enumerate(['ethos', 'pathos', 'logos']):
        sns.heatmap(pivot_table[score_type], annot=True, cmap='coolwarm', ax=axes[i], fmt='.2f')
        axes[i].set_title(f'{score_type.capitalize()} by Triad and Persona')
        axes[i].set_xlabel('Triad')
        axes[i].set_ylabel('Persona')

    plt.tight_layout()
    return plt.gcf()

In [ ]:
##Description: To filter out comments such as using a slider so the user can see the difference of 10 comments or 1000 comments in each graph.

def filter_comments(df, n):
    # Filter the DataFrame based on the number of comments
    return df.head(n)

#num_comments = st.slider('Select number of comments to display', min_value=1, max_value=len(comment_df1), value=len(comment_df1))
#filtered_df = filter_comments(comment_df1, num_comments)

In [ ]:
##Description: Creates a sunburst chart by taking in a dataframe, the type of EPL score to be displayed, and the number of comments for which data will be displayed.
##Shows distribution of one EPL score at a time across all triads and personas.

def triadSunburst(df, type, num):
   color_map = {
       'mind': '#C9E4AF',
       'heart': '#FFD3AB',
       'gut': '#ADD2F0'

   }
   fig = px.sunburst(
       df.head(num),
       path=['triad', 'personas'],
       values=type,
       color='triad',
       color_discrete_map=color_map
   )

   return fig

In [ ]:
##Description: Creates a parallel coordinates graph by taking in a dataframe and the number of comments for which data will be displayed.
##Shows trend for EPL scores depending on which triad a comment is classified in.

def eplParallelCoords(df, num):
   df = df.head(num)
   df['triad'] = df['triad'].astype('category')
   color_map = {
      'mind': '#C9E4AF',
      'heart': '#FFD3AB',
      'gut': '#ADD2F0'

   }
   selection = alt.selection_single(fields=['triad'], bind='legend', name='Select Triad')

   fig = alt.Chart(df).transform_window(
       index='count()'
   ).transform_fold(
       ['ethos', 'pathos', 'logos'],
       as_=['score type', 'score value']
   ).mark_line().encode(
       x='score type:N',
       y='score value:Q',
       color=alt.Color('triad:N', scale=alt.Scale(domain=list(color_map.keys()), range=list(color_map.values()))),
       detail='index:N',
       opacity=alt.condition(selection, alt.value(0.5), alt.value(0.1))
   ).add_selection(
       selection
   ).properties(width=700)

   return fig

In [ ]:
##Description: Creates a bubble chart, where the bigger the bubble, the bigger the logos. The legends will be the personas as the user can click on the legends and filter out what persona they want.
##Show the trend between numeric variables of ethos, pathos, logos. It’s more 3-dimensional as the size helps it show a third variable.



def epl_bubble_chart(df):
    fig = px.scatter(
        df,
        x='ethos',
        y='pathos',
        size='number_of_comments',
        color='persona',
        hover_name='comment_id',
        hover_data={
            'ethos': True,
            'pathos': True,
            'logos': True,
            'number_of_comments': True,
            'comment_text': True
        },
        title="Bubble Chart of EPL Scores",
        labels={'ethos': 'Ethos Score', 'pathos': 'Pathos Score', 'number_of_comments': 'Number of Comments'}
    )
    return fig

# selected_comment_id = st.selectbox(
#     'Select a Comment ID to view details:',
#     options=df['comment_id'].tolist()
# )

# if selected_comment_id:
#     selected_row = df[df['comment_id'] == selected_comment_id].iloc[0]
#     st.write(f"**Details for Comment ID: {selected_comment_id}**")
#     st.write(f"**Comment Text:** {selected_row['comment_text']}")
#     st.write(f"**Ethos Score:** {selected_row['ethos']}")
#     st.write(f"**Pathos Score:** {selected_row['pathos']}")
#     st.write(f"**Logos Score:** {selected_row['logos']}")
#     # place holder for EPL explanation and author, etc

In [ ]:
##Description: Makes a violin plot with all personas as legends but the x-axis are EPL and y-axis are scores.
##Same as before: when you click on the legends, you can filter out the personas.

def epl_violin_plot(df):
    df_melted = df.melt(id_vars=['personas'], value_vars=['ethos', 'pathos', 'logos'], var_name='score_type', value_name='score')
    fig = px.violin(df_melted, y='score', x='score_type', color='personas', box=True, points='all', title="Violin Plot of EPL Scores by Persona")
    return fig

In [ ]:
##Description: Makes a Scatter Matrix of EPL scores with different color dots for each persona

def st_scatter_matrix(df, use_container_width: bool):
    import altair as alt


    chart = alt.Chart(df).mark_circle().encode(
        alt.X(alt.repeat("column"), type='quantitative'),
        alt.Y(alt.repeat("row"), type='quantitative'),
        color='personas:N'
    ).properties(
        width=100,
        height=100
    ).repeat(
        row=['ethos', 'pathos', 'logos'],
        column=['logos', 'pathos', 'ethos']
    ).interactive()

    tab1, tab2 = st.tabs(["Streamlit theme (default)", "Altair native theme"])

    with tab1:
        st.altair_chart(chart, theme="streamlit", use_container_width=True)
    with tab2:
        st.altair_chart(chart, theme=None, use_container_width=True)